In [ ]:
"""
PREPROCESSING PIPELINE - DELIVERY DELAY PROJECT

Mô tả:
Pipeline này xử lý dữ liệu thô từ nhiều file CSV theo từng giai đoạn thời gian 
(Apr-Jun dùng làm train, Jul-Sep dùng làm test - out-of-time validation).

Các bước chính:
1. Merge dữ liệu delay / non-delay thành train và test set
2. Chuẩn hóa tên cột (loại bỏ khoảng trắng, thống nhất format)
3. Loại bỏ:
   - Data leakage columns (REASON_CD, SHIP_DECISION_NO, ...)
   - Các cột không cần thiết
4. Xử lý missing values cho categorical và flag variables
5. Ép kiểu dữ liệu:
   - Datetime: Order_date, VSD
   - Category: PRODUCT_CD, SUPPLIER_CD, ...
6. Đảm bảo train/test có cùng schema
7. Lưu dữ liệu sạch dưới dạng Parquet để phục vụ modeling

Output:
- train_cleaned.parquet
- test_cleaned.parquet
"""

# Unzip file

In [15]:
# import zipfile

# # path to the zip file
# zip_path = r'D:\Ds108\Lab4\Data for practice-20260417T070843Z-3-001.zip'
# # directory where you want to extract the files
# extract_path = r'D:\Ds108\Lab4\raw_data'

# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_path)


# Preprocess

In [16]:
# ==============================================================================
# BƯỚC 1 / INSTRUCTION LIST: RAW TO TIDY
# Mục tiêu: Làm sạch dữ liệu rò rỉ, xử lý missing và tối ưu hóa kiểu dữ liệu
# ==============================================================================

import pandas as pd
import numpy as np
import warnings
import os
warnings.filterwarnings('ignore')

# -----------------------------------------------------------
# 1. DATA LOADING (Phân chia Tập A và Tập B theo yêu cầu)
# -----------------------------------------------------------
BASE_PATH = r'D:\Ds108\Lab4\raw_data\Data for practice'

# Tập A: Giai đoạn 4-6
df_A_delay = pd.read_csv(f'{BASE_PATH}\delay_4_6_CONDITION_PRODUCT_SUPPLIER.csv')
df_A_notdelay = pd.read_csv(f'{BASE_PATH}\\not_delay_4_6_CONDITION_PRODUCT_SUPPLIER.csv')
df_A = pd.concat([df_A_delay, df_A_notdelay], ignore_index=True)

# Tập B: Giai đoạn 7-9
df_B_delay = pd.read_csv(f'{BASE_PATH}\delay_7_9_CONDITION_PRODUCT_SUPPLIER.csv')
df_B_notdelay = pd.read_csv(f'{BASE_PATH}\\not_delay_7_9_CONDITION_PRODUCT_SUPPLIER.csv')
df_B = pd.concat([df_B_delay, df_B_notdelay], ignore_index=True)

# Lấy các cột chung
common_cols = df_A.columns.intersection(df_B.columns)
df_A = df_A[common_cols].copy()
df_B = df_B[common_cols].copy()

print(f"Shape Tap A (Thang 4-6): {df_A.shape}")
print(f"Shape Tap B (Thang 7-9): {df_B.shape}")




Shape Tap A (Thang 4-6): (399053, 37)
Shape Tap B (Thang 7-9): (1074897, 37)


In [17]:
# -----------------------------------------------------------
# 2. DATA CLEANING PIPELINE
# -----------------------------------------------------------
def raw_to_tidy_pipeline(df):
    df_cp = df.copy()
    
    # Chuẩn hóa tên cột
    cols = pd.Series(df_cp.columns).astype(str).str.strip().str.replace(r'\s+', '_', regex=True)
    df_cp.columns = cols
    df_cp = df_cp.loc[:, ~df_cp.columns.duplicated()]
    
    # A. DROP LEAKAGE & NOISE (Loại bỏ nhiễu và rò rỉ tương lai)
    leakage_cols = ['REASON_CD', 'SOUF_RCV_NO', 'QTUF_RCV_NO', 'SHIP_DECISION_NO']
    noise_cols = ['SUBSIDIARY_CD', 'INNER_CD', 'GLOBAL_NO', 'Sales_order_line_number'] # Đã xóa unique ID
    df_cp = df_cp.drop(columns=[c for c in (leakage_cols + noise_cols) if c in df_cp.columns], errors='ignore')
    
    # B. MISSING VALUE TREATMENT
    if 'Ship_Mode' in df_cp.columns:
        df_cp['Ship_Mode'] = df_cp['Ship_Mode'].fillna('Missing')
    if 'SUPPLIER_DIV' in df_cp.columns:
        df_cp['SUPPLIER_DIV'] = df_cp['SUPPLIER_DIV'].fillna(-1)
    if 'OTHER_AREA_SHIP_DIV' in df_cp.columns:
        df_cp['OTHER_AREA_SHIP_DIV'] = pd.to_numeric(df_cp['OTHER_AREA_SHIP_DIV'].replace(' ', np.nan), errors='coerce').fillna(0)
    if 'Consider_count_hodiday_Saturday' in df_cp.columns:
        df_cp['Consider_count_hodiday_Saturday'] = pd.to_numeric(df_cp['Consider_count_hodiday_Saturday'], errors='coerce').fillna(0)
        
    # C. DATA TYPE CASTING (Ép kiểu cho Tree-based model)
    df_cp['Order_date'] = pd.to_datetime(df_cp['Order_date'], errors='coerce')
    df_cp['VSD'] = pd.to_datetime(df_cp['VSD'], errors='coerce')
    
    categorical_cols = ['PRODUCT_CD', 'BRAND_CD', 'SUPPLIER_CD', 'Ship_Mode', 'DELI_DIV', 'PACKING_RANK', 'Stock_class']
    for col in categorical_cols:
        if col in df_cp.columns:
            df_cp[col] = df_cp[col].astype(str).astype('category')
            
    return df_cp

print("\nCleaning Tap A...")
df_A_clean = raw_to_tidy_pipeline(df_A)

print("Cleaning Tap B...")
df_B_clean = raw_to_tidy_pipeline(df_B)





Cleaning Tap A...
Cleaning Tap B...


In [18]:
# -----------------------------------------------------------
# 3. SAVING TIDY DATA
# -----------------------------------------------------------
processed_dir = r'D:\Ds108\Lab4\processed_data'
if not os.path.exists(processed_dir):
    os.makedirs(processed_dir)

df_A_clean.to_parquet(f'{processed_dir}\\dataset_A_tidy.parquet', index=False)
df_B_clean.to_parquet(f'{processed_dir}\\dataset_B_tidy.parquet', index=False)

print("\n=> [Hoan Thanh] Data da dat chuan Tidy. San sang cho Deep EDA!")


=> [Hoan Thanh] Data da dat chuan Tidy. San sang cho Deep EDA!
